# Notebook 06: Quantum NeuroAI — From Organoid Spikes to Density Matrices

**QCCCM** — Quantum Compute for Computational Cognitive Modeling

---

## What you'll learn

1. How to convert **neural spike train data** into quantum density matrices
2. **Qubit resource estimation** for biological neural systems (Wolff et al. 2025)
3. **Quantum information-theoretic analysis** of neural population activity
4. How **von Neumann entropy** and **state fidelity** reveal dynamics invisible to classical metrics
5. The BL-1 / NWB organoid data pipeline into QCCCM

### The key idea

A population of N neurons with binary firing states maps to an N-qubit quantum system. The density matrix captures:
- **Diagonal**: firing pattern probabilities (classical statistics)
- **Off-diagonal**: correlations and coherences between neurons

This is not a claim that neurons are quantum — it's that the **mathematical framework of quantum information** provides strictly more powerful tools for analysing neural population data than classical statistics alone (Schneidman et al. 2006, Wolff et al. 2025).

### Data sources
- **BL-1 cortical culture simulator** (synthetic ground truth)
- **NWB organoid recordings** from DANDI Archive (real data)
- **Synthetic spike trains** (for demonstration)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

from qcccm.neuroai.resource_estimation import (
    NEURAL_SYSTEMS, estimate_neural_qubits, full_resource_table, print_resource_table,
)
from qcccm.neuroai.data_interface import (
    spike_raster_to_rates, neural_data_to_density_matrix,
    quantum_neural_analysis, neural_state_fidelity_over_time,
)
from qcccm.neuroai.neural_states import (
    firing_rates_to_density_matrix, neural_entropy,
)
from qcccm.core.density_matrix import purity, von_neumann_entropy

plt.rcParams['figure.dpi'] = 120
print('QCCCM NeuroAI pipeline loaded')

## 1. Qubit Resource Estimation

Wolff et al. (2025) Fig. 3: biological systems mapped to qubit requirements.
Amplitude encoding gives exponential compression — 86 billion neurons need only 37 qubits.

In [ ]:
print_resource_table()

# Key practical insight
for key in ['hd_mea_recording', 'organoid_mature', 'human']:
    est = estimate_neural_qubits(NEURAL_SYSTEMS[key])
    print(f'{est.system}: {est.n_neurons:,} neurons -> {est.amplitude_qubits} qubits')

## 2. Synthetic Cortical Culture Data

We simulate 20 neurons in 4 subgroups with correlated bursting,
mimicking organoid network dynamics.

In [ ]:
# Generate synthetic cortical culture data
np.random.seed(42)
n_neurons = 20
duration_s = 10.0
dt_s = 0.001
n_steps = int(duration_s / dt_s)

group_rates = [5.0, 15.0, 8.0, 25.0]
group_sizes = [5, 5, 5, 5]

raster = np.zeros((n_steps, n_neurons), dtype=np.float32)
for t in range(n_steps):
    time_s = t * dt_s
    burst_prob = 0.3 * np.exp(-((time_s % 2.0) - 1.0)**2 / 0.02)
    idx = 0
    for rate, size in zip(group_rates, group_sizes):
        for n in range(size):
            if np.random.random() < (rate + burst_prob * 50.0) * dt_s:
                raster[t, idx] = 1
            idx += 1

rates = spike_raster_to_rates(raster, dt_ms=1.0, bin_ms=100.0)
print(f'Raster: {raster.shape}, Rates: {rates.shape}')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for n in range(n_neurons):
    spk = np.where(raster[:, n] > 0)[0] * dt_s
    axes[0].plot(spk, np.ones_like(spk) * n, '|', markersize=2, color='black', alpha=0.5)
axes[0].set_ylabel('Neuron')
axes[0].set_title('Spike Raster')
axes[1].plot(np.arange(rates.shape[0]) * 0.1, rates.mean(axis=1), 'b-')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Mean rate')
plt.tight_layout()
plt.show()

## 3. Spikes to Density Matrices

Select 4 neurons (one per group) and construct density matrices over time.
Correlations appear as off-diagonal elements.

In [ ]:
neuron_indices = [0, 5, 10, 15]
time_points = [10, 30, 50, 70, 90]
fig, axes = plt.subplots(1, len(time_points), figsize=(16, 3))

for ax, t in zip(axes, time_points):
    rho = neural_data_to_density_matrix(rates, neuron_indices=neuron_indices, time_bin=t)
    im = ax.imshow(np.abs(np.asarray(rho)), cmap='Blues', vmin=0)
    ax.set_title(f't={t*0.1:.1f}s\nS={float(von_neumann_entropy(rho)):.2f}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.8, label='|rho_ij|')
plt.suptitle(f'Density Matrices Over Time (neurons {neuron_indices})', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Full quantum analysis
analysis = quantum_neural_analysis(rates, neuron_indices=neuron_indices)
times = np.array(analysis['time_bins']) * 0.1

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(times, analysis['entropy'], 'b-o', markersize=4, linewidth=1.5)
axes[0].set_ylabel('S(rho)')
axes[0].set_title('Von Neumann Entropy')
axes[1].plot(times, analysis['purity'], 'r-o', markersize=4, linewidth=1.5)
axes[1].set_ylabel('Tr(rho^2)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Purity')
plt.suptitle('Quantum Information Dynamics', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. State Fidelity: Detecting Transitions

Fidelity F(rho_t, rho_{t+1}) measures state change. Drops indicate bursts,
state transitions, or information processing events — sensitive to correlation
structure changes, not just rate changes.

In [ ]:
times_fid, fidelities = neural_state_fidelity_over_time(
    rates, neuron_indices=neuron_indices, bin_step=1,
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(np.arange(rates.shape[0]) * 0.1, rates[:, neuron_indices].mean(axis=1), 'b-', alpha=0.7)
axes[0].set_ylabel('Mean rate')
axes[0].set_title('Classical: Firing Rate')
axes[1].plot(times_fid * 0.1, fidelities, 'r-')
axes[1].axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('F(rho_t, rho_{t+1})')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Quantum: State Fidelity')
for d in (times_fid * 0.1)[fidelities < 0.9][:10]:
    axes[0].axvline(x=d, color='red', alpha=0.2)
    axes[1].axvline(x=d, color='red', alpha=0.2)
plt.suptitle('Fidelity Drops Detect State Transitions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f'Transitions detected (F<0.9): {np.sum(fidelities < 0.9)}')

## 5. Loading Real Data

```python
# BL-1 NWB loader
from bl1.validation.loaders import load_nwb_spike_trains
data = load_nwb_spike_trains('path/to/organoid.nwb')
from qcccm.neuroai.data_interface import spike_trains_to_rates
rates, _ = spike_trains_to_rates(data['spike_times'], data['duration_s'])

# DANDI streaming (neurojax)
from neurojax.io.dandi import DandiLoader
loader = DandiLoader('001603')  # van der Molen 2025
nwbfile, io = loader.stream_nwb(loader.list_files()[0])

# Then: same pipeline
rho = neural_data_to_density_matrix(rates, neuron_indices=[0,1,2,3])
analysis = quantum_neural_analysis(rates)
```

Available organoid datasets:
- **Sharf 2022** — HD-MEA organoid (Zenodo)
- **van der Molen 2025** — Protosequences (DANDI 001603)
- **Braingeneers SpikeCanvas** — Large-scale (DANDI 001747)

---

## Exercises

**6.1:** Vary correlation strength from 0 to 0.8 and plot von Neumann entropy.

**6.2:** Compare Shannon entropy of firing rates with von Neumann entropy of density matrices. When do they diverge?

**6.3 (Real data):** Download Sharf 2022 and run the pipeline during drug application vs baseline.

**6.4 (BL-1):** Run a 30s BL-1 simulation and track quantum observables as the network matures.

## References

- Wolff et al. (2025). Quantum Computing for Neuroscience. OSF Preprints.
- Schneidman et al. (2006). Weak pairwise correlations. *Nature*.
- Sharf et al. (2022). Functional neuronal circuitry in organoids. *Nat Commun*.
- van der Molen et al. (2026). Protosequences in organoids. *Nat Neurosci*.